# Fooling a machine learning model

A model can score very well on test data and still be fooled by inputs chosen to fool it. These inputs are called **adversarial examples**, and using them against a deployed model is an **evasion attack**.

You will fool a company's digit reader in three ways: with ordinary changes that a person would barely notice, by using the model's own gradients, and then by making it give the exact wrong answer you choose.

*This lab is adapted from the [Security-of-AI](https://github.com/habroshan/Security-of-AI) project by Anglia Ruskin University, funded by CyBOK. Contains public sector information licensed under the Open Government Licence v3.0.*

## 1. Set up

Run this cell. It also defines small helpers for talking to the model.

In [ ]:
import json
import os

import numpy as np
import requests
import matplotlib.pyplot as plt
from scipy import ndimage

API_URL = "http://hackerbot:5000"


def get_challenge(name):
    challenge = requests.get(f"{API_URL}/challenge/{name}").json()
    challenge["image"] = np.array(challenge["image"], dtype="float32")
    return challenge


def ask_model(image):
    """Returns the model's chosen digit and its 10 probabilities."""
    answer = requests.post(f"{API_URL}/predict", json={"image": image.tolist()}).json()
    return answer["prediction"], np.array(answer["full_probabilities"])


def show(image, title=""):
    plt.figure(figsize=(2, 2))
    plt.imshow(image, cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 2. Ordinary changes

Your first challenge is an image of a digit. Look at it, and see what the model makes of it.

In [ ]:
challenge = get_challenge("transform")
image = challenge["image"]
prediction, probabilities = ask_model(image)
show(image, f"Model reads: {prediction}")
print("Allowed rotation (degrees either way):", challenge["max_angle"])
print("Allowed shift (pixels each way):", challenge["max_shift"])

This function rotates and shifts an image. Try it on the image above.

**Your task.** Find a rotation and shift, within the limits, that make the model misread the digit. You can try values by hand, or write a loop that tries a range of them. Put the values you find in the three variables, and check the result.

In [ ]:
def transform(image, angle, shift_x, shift_y):
    turned = ndimage.rotate(image, angle, reshape=False, order=1)
    moved = ndimage.shift(turned, (shift_y, shift_x), order=1)
    return np.clip(moved, 0.0, 1.0)


angle = 0    # TODO: find a rotation that works
shift_x = 0  # TODO: and a shift
shift_y = 0  # TODO

changed = transform(image, angle, shift_x, shift_y)
prediction, _ = ask_model(changed)
show(changed, f"Model reads: {prediction}")

Run this, then say `ready` to Hackerbot.

In [ ]:
with open(os.path.expanduser("~/evasion_transform.json"), "w") as handle:
    json.dump({"angle": angle, "shift_x": shift_x, "shift_y": shift_y}, handle)
print("Saved")

## 3. Using the model's gradient

When a model is trained, a number called the **loss** measures how wrong it is. Training nudges the model's weights to make the loss smaller.

An attacker can turn this around. The **gradient** of the loss with respect to the *input image* says, for every pixel, which way to change that pixel to make the model *more* wrong. The company's API will give you this gradient for an image and a label.

The **fast gradient sign method** (FGSM) takes one step: every pixel moves by the same small amount, called **epsilon**, in the direction the gradient says will increase the loss. `np.sign` turns each value of the gradient into +1 or -1, which is just its direction.

Look at the gradient for the next challenge. It looks like noise to us, but it is telling you exactly how to change the image.

In [ ]:
def get_gradient(image, label):
    """The gradient of the model's loss, for this image and this label."""
    answer = requests.post(f"{API_URL}/gradient", json={"image": image.tolist(), "label": int(label)}).json()
    return np.array(answer["gradient"], dtype="float32")


challenge = get_challenge("fgsm")
image, true_label, epsilon = challenge["image"], challenge["true_label"], challenge["epsilon"]
prediction, _ = ask_model(image)
show(image, f"Model reads: {prediction}")
print("Epsilon:", epsilon)

gradient = get_gradient(image, true_label)
plt.figure(figsize=(2, 2))
plt.imshow(gradient, cmap="gray")
plt.axis("off")
plt.show()

**Your task.** Make `adversarial` by taking a step of size `epsilon` in the direction of `np.sign(gradient)`. Then use `np.clip` to keep the pixel values between 0 and 1.

In [ ]:
adversarial = None  # TODO: apply FGSM to image

prediction, _ = ask_model(adversarial)
show(adversarial, f"Model reads: {prediction}")
print("Largest pixel change:", np.abs(adversarial - image).max())

Run this, then say `ready` to Hackerbot.

In [ ]:
np.save(os.path.expanduser("~/evasion_fgsm.npy"), adversarial)
print("Saved")

## 4. Choosing the wrong answer

So far you only needed the model to be wrong. A more capable attacker chooses *which* wrong answer the model gives.

**Projected gradient descent** (PGD) takes many small steps instead of one. To reach a target, ask for the gradient for the **target** label, and step in the opposite direction, because you want the loss for that label to go *down*. After every step you "project": you clip the image so that no pixel has moved more than epsilon from the original, and clip to 0-1 as before.

**Your task.** Complete the loop below. Use a step size of `epsilon / 10`, for 40 steps.

In [ ]:
challenge = get_challenge("pgd")
image, target, epsilon = challenge["image"], challenge["target"], challenge["epsilon"]
print("Make the model read this as:", target, " Epsilon:", epsilon)

adversarial = image.copy()
step_size = epsilon / 10

for step in range(40):
    gradient = get_gradient(adversarial, target)
    # TODO: move adversarial by step_size against the sign of the gradient
    # TODO: clip it to within epsilon of the original image
    # TODO: clip it to between 0 and 1

prediction, _ = ask_model(adversarial)
show(adversarial, f"Model reads: {prediction}")
print("Largest pixel change:", np.abs(adversarial - image).max())

Run this, then say `ready` to Hackerbot.

In [ ]:
np.save(os.path.expanduser("~/evasion_pgd.npy"), adversarial)
print("Saved")